# Reference-data generation (pyscf / gpu4pyscf)

Folder-staged pipeline: each stage cell drains its input folder into the next, so a
molecule advances simply by running cells top-to-bottom. Stages are **idempotent and
resumable** — re-running a cell skips items already produced, so a Colab reconnect just
means re-running the cells.

```
00_inbox  ->  01_optimized  ->  02_frequencies  ->  03_wigner  ->  04_labeled
```

Per molecule: optimize (wB97M-V/def2-svpd) → harmonic frequencies → Wigner sampling
(300 K, 500 structures) → label each with energy/forces/dipole/quadrupole/
polarizability/dipole-derivatives → extended-XYZ matching `data/labels/*.extxyz`.

The compute core prefers **gpu4pyscf** (Colab GPU) and falls back to **pyscf** (local CPU).

## 1. Install
On Colab (GPU) this installs the gpu4pyscf stack; locally it installs the CPU pyscf backend.
Either way it then installs the `rsfff` package in editable mode.

In [4]:
import sys, subprocess, os, importlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # GPU stack (mirrors notebooks/almo_eda_pyscf.ipynb) + geomeTRIC for opt.
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "pyscf==2.8.0", "basis-set-exchange==0.11", "geometric==1.1.0",
        "cupy-cuda12x==13.4.1", "cutensor-cu12==2.2.0",
        "gpu4pyscf-cuda12x==1.7.6"], check=True)

    # Get the repo and *always sync it to the latest main*. A leftover clone from
    # an earlier session can be stale (missing rsfff.qcgen); refreshing here avoids
    # the "No module named 'rsfff.qcgen'" trap that skipping the clone would cause.
    REPO_DIR, REPO_URL = "rsfff", "https://github.com/heindelj/rsfff.git"
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    print("rsfff repo at commit:", head)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
else:
    # Local CPU backend. VV10 (needed by wB97M-V) is built into pyscf; we do NOT
    # install pyscf-dispersion (D3/D4 only, and its release is broken vs new numpy).
    subprocess.run([sys.executable, "-m", "pip", "install", "pyscf>=2.14", "geometric"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."], check=True)

# Drop any stale cached import of a previously-installed rsfff so the next cell
# picks up the version we just installed (no kernel restart needed).
for _m in [k for k in list(sys.modules) if k == "rsfff" or k.startswith("rsfff.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
print("install cell done; IN_COLAB =", IN_COLAB)

rsfff repo at commit: 7088f76
install cell done; IN_COLAB = True


## 2. Setup
Pick the pipeline root (Colab Drive folder or a local directory), create the stage folders,
and report which backend is active.

In [5]:
import os
# macOS libomp guard (harmless elsewhere); must be set before importing pyscf.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path
from rsfff.qcgen import pipeline as pl
from rsfff.qcgen.backend import backend_name

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/QC_runs")
else:
    # Local: keep runs under the repo's data/ tree.
    ROOT = Path("..") / "data" / "qc_runs"

cfg = pl.PipelineConfig(
    root=ROOT,
    xc="wB97M-V",
    basis="def2-svpd",
    temperature=300.0,
    n_samples=500,
    seed=0,
)
pl.make_dirs(cfg)
print("backend :", backend_name())
print("root    :", cfg.root.resolve())
print("method  :", cfg.xc, "/", cfg.basis, "| T =", cfg.temperature, "K | N =", cfg.n_samples)

ModuleNotFoundError: No module named 'rsfff.qcgen'

## 3. Seed the inbox
Copy the molecules you want to process into `00_inbox/`. Charges/multiplicities are looked up
in `pipeline.SPECIES` (open-shell allowed, e.g. `oh_radical` → doublet). Edit `NAMES` to choose
which species to run; `None` seeds everything in `data/mols/`.

In [ ]:
import rsfff

# Locate the built-in reference geometries from the *installed package*, not a
# CWD-relative path -- works both locally (notebooks/) and on Colab (/content),
# where the repo is cloned to /content/rsfff.
REPO = Path(rsfff.__file__).resolve().parents[1]
MOLS = REPO / "data" / "mols"

NAMES = ["h2o"]  # subset to run; None = every *.xyz in data/mols

if NAMES is None:
    xyz_paths = sorted(MOLS.glob("*.xyz"))
else:
    xyz_paths = [MOLS / f"{n}.xyz" for n in NAMES]

seeded = pl.seed_inbox(cfg, xyz_paths)
print(f"seeded {len(seeded)} file(s) into {cfg.root / pl.INBOX}:")
print(" ", ", ".join(seeded) if seeded else "(nothing new; inbox already populated)")
# You can also drop your own .xyz files directly into the inbox folder above and
# the stages will pick them up. On Colab that folder lives on Google Drive
# (ROOT = /content/drive/MyDrive/QC_runs), so inputs and outputs both persist
# across sessions -- no separate Drive plumbing needed.

## 4. Stage: optimize geometries
`00_inbox/*.xyz` → `01_optimized/<name>.xyz` (+ `<name>.json` meta).

In [ ]:
pl.run_stage(pl.STAGE_OPTIMIZE, cfg)

[09:55:33] stage 'optimize' [pyscf]: 1 candidate(s) in 00_inbox/
[09:55:33]   optimize: h2o ...


geometric-optimize called with the following command line:
/opt/miniconda3/envs/rsfff/lib/python3.11/site-packages/ipykernel_launcher.py --f=/Users/joseph.heindel/Library/Jupyter/runtime/kernel-v36ea90f29de0834577aa73908fdd460b7bae1649b.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%

[09:56:21]   optimize: h2o done (48.5s)
[09:56:21] stage 'optimize': 1 done, 0 skipped, 0 failed


(['h2o'], [], [])

## 5. Stage: harmonic frequencies
`01_optimized/` → `02_frequencies/<name>.npz` (Hessian, masses, geometry). Analytic Hessian
where supported; automatic finite-difference fallback for UKS + NLC (open-shell wB97M-V).

In [ ]:
pl.run_stage(pl.STAGE_FREQUENCIES, cfg)

[09:56:21] stage 'frequencies' [pyscf]: 1 candidate(s) in 01_optimized/
[09:56:21]   frequencies: h2o ...
[09:58:41]   frequencies: h2o done (139.6s)
[09:58:41] stage 'frequencies': 1 done, 0 skipped, 0 failed


(['h2o'], [], [])

## 6. Stage: Wigner sampling
`02_frequencies/` → `03_wigner/<name>.extxyz` (`n_samples` geometries at `temperature`).

In [ ]:
pl.run_stage(pl.STAGE_WIGNER, cfg)

[09:58:41] stage 'wigner' [pyscf]: 1 candidate(s) in 02_frequencies/
[09:58:41]   wigner: h2o ...
[09:58:41]   wigner: h2o done (0.0s)
[09:58:41] stage 'wigner': 1 done, 0 skipped, 0 failed


(['h2o'], [], [])

## 7. Stage: label properties
`03_wigner/` → `04_labeled/<name>.extxyz` (energy, forces, dipole, quadrupole,
polarizability, dipole derivatives). On success the original inbox `.xyz` is archived to
`_completed/`. This is the expensive stage; it streams frames to disk and writes atomically.

In [ ]:
pl.run_stage(pl.STAGE_LABEL, cfg)

[09:58:41] stage 'label' [pyscf]: 1 candidate(s) in 03_wigner/
[09:58:41]   label: h2o ...


KeyboardInterrupt: 

## 8. Collect + summary
Concatenate all labeled species into one dataset file and report per-species frame counts and
any failures (tracebacks live in `_failed/`).

In [ ]:
out_path, counts = pl.collect(cfg)
print("combined dataset ->", out_path.resolve())
for name, n in sorted(counts.items()):
    print(f"  {name:14s} {n} frames")

failed = sorted((cfg.root / pl.FAILED).glob("*.error.log"))
if failed:
    print("\nfailures:")
    for f in failed:
        print("  ", f.name)